In [2]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

import ast
import time

In [3]:
ratings = pd.read_csv(
    "../data/processed/interactions_clean.csv"
)

recipes_clean = pd.read_csv(
    "../data/processed/recipes_clean.csv"
)

print("Datasets loaded successfully!")
print("Interactions:", ratings.shape)
print("Recipes:", recipes_clean.shape)

Datasets loaded successfully!
Interactions: (1071520, 5)
Recipes: (231637, 15)


In [4]:
ratings = ratings[ratings["rating"] > 0].copy()

ratings = ratings.reset_index(drop=True)

print("Total valid ratings:", len(ratings))
print("Unique users:", ratings["user_id"].nunique())
print("Unique recipes:", ratings["recipe_id"].nunique())

Total valid ratings: 1071520
Unique users: 196098
Unique recipes: 226590


In [5]:
user_counts = ratings["user_id"].value_counts()

eligible_users = user_counts[
    user_counts >= 10
].index

evaluation_ratings = ratings[
    ratings["user_id"].isin(eligible_users)
].copy()

selected_users = evaluation_ratings[
    "user_id"
].drop_duplicates().sample(
    n=min(1000, evaluation_ratings["user_id"].nunique()),
    random_state=42
)

evaluation_ratings = evaluation_ratings[
    evaluation_ratings["user_id"].isin(selected_users)
].copy()

print("Selected users:", evaluation_ratings["user_id"].nunique())
print("Selected interactions:", len(evaluation_ratings))

Selected users: 1000
Selected interactions: 65896


In [6]:
test_ratings = evaluation_ratings.groupby(
    "user_id", group_keys=False
).sample(
    frac=0.2,
    random_state=42
)

train_ratings = evaluation_ratings.drop(
    test_ratings.index
)

print("Training interactions:", len(train_ratings))
print("Testing interactions:", len(test_ratings))

Training interactions: 52736
Testing interactions: 13160


In [7]:
user_ids = train_ratings["user_id"].unique()
recipe_ids = train_ratings["recipe_id"].unique()

user_map = {
    user: i for i, user in enumerate(user_ids)
}

recipe_map = {
    recipe: i for i, recipe in enumerate(recipe_ids)
}

row = train_ratings["user_id"].map(user_map)
col = train_ratings["recipe_id"].map(recipe_map)

sparse_matrix = csr_matrix(
    (
        train_ratings["rating"].astype(np.float32),
        (row, col)
    ),
    shape=(len(user_ids), len(recipe_ids))
)

print("Sparse matrix created!")
print("Matrix shape:", sparse_matrix.shape)
print("Non-zero ratings:", sparse_matrix.nnz)

Sparse matrix created!
Matrix shape: (1000, 37133)
Non-zero ratings: 52736


In [8]:
from sklearn.neighbors import NearestNeighbors

item_user_matrix = sparse_matrix.T.tocsr()

cf_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21,
    n_jobs=-1
)

start_time = time.time()

cf_model.fit(item_user_matrix)

print("Collaborative Filtering model trained!")
print("Items:", item_user_matrix.shape[0])
print("Time taken:", round(time.time() - start_time, 2), "seconds")

Collaborative Filtering model trained!
Items: 37133
Time taken: 0.11 seconds


In [9]:
distances, neighbor_indices = cf_model.kneighbors(
    item_user_matrix,
    n_neighbors=21
)

print("Similar recipes calculated!")
print("Neighbor matrix shape:", neighbor_indices.shape)

Similar recipes calculated!
Neighbor matrix shape: (37133, 21)


In [10]:
test_relevant = test_ratings[
    test_ratings["rating"] >= 4
]

test_user_items = (
    test_ratings.groupby("user_id")["recipe_id"]
    .apply(set)
    .to_dict()
)

print("Relevant test interactions:", len(test_relevant))
print("Users with test interactions:", len(test_user_items))

Relevant test interactions: 12472
Users with test interactions: 1000


In [11]:
def recommend_cf(user_id, n=10):

    if user_id not in user_map:
        return []

    user_index = user_map[user_id]

    user_ratings = sparse_matrix[user_index].toarray().flatten()

    rated_indices = np.where(user_ratings > 0)[0]

    scores = np.zeros(len(recipe_ids), dtype=float)

    for recipe_index in rated_indices:

        rating = user_ratings[recipe_index]

        similar_indices = neighbor_indices[recipe_index][1:]
        similar_distances = distances[recipe_index][1:]

        similarities = 1 - similar_distances

        scores[similar_indices] += rating * similarities

    scores[rated_indices] = -1

    top_indices = np.argsort(scores)[::-1][:n]

    return [
        recipe_ids[index]
        for index in top_indices
        if scores[index] > 0
    ]

In [12]:
def calculate_metrics(recommendation_function, k=10):

    precisions = []
    recalls = []

    for user_id in test_user_items:

        recommendations = recommendation_function(user_id, n=k)

        actual_relevant = test_relevant[
            test_relevant["user_id"] == user_id
        ]["recipe_id"].tolist()

        if len(actual_relevant) == 0:
            continue

        recommended_set = set(recommendations)
        relevant_set = set(actual_relevant)

        hits = len(recommended_set & relevant_set)

        precision = hits / k
        recall = hits / len(relevant_set)

        precisions.append(precision)
        recalls.append(recall)

    return {
        "Precision@10": round(np.mean(precisions), 4),
        "Recall@10": round(np.mean(recalls), 4)
    }

In [13]:
cf_results = calculate_metrics(recommend_cf)

print("Collaborative Filtering Results:")
print(cf_results)

Collaborative Filtering Results:
{'Precision@10': np.float64(0.0003), 'Recall@10': np.float64(0.0003)}


In [14]:
test_user = test_relevant["user_id"].iloc[0]

print("Test User:", test_user)

print("\nActual relevant recipes:")
print(
    test_relevant[
        test_relevant["user_id"] == test_user
    ][["recipe_id", "rating"]].head(10)
)

print("\nRecommended recipes:")
print(recommend_cf(test_user, n=10))

Test User: 4862

Actual relevant recipes:
        recipe_id  rating
13519      100243       5
826856      21541       5
653552     137345       5
46668       14723       5

Recommended recipes:
[np.int64(184359), np.int64(35011), np.int64(251451), np.int64(82474), np.int64(331056), np.int64(11860), np.int64(17762), np.int64(125515), np.int64(466), np.int64(124476)]


In [15]:
print("Users with relevant test recipes:", len(test_relevant["user_id"].unique()))

print("Average relevant recipes per user:",
      round(test_relevant.groupby("user_id").size().mean(), 2))

print("Average recommendations per user:",
      round(
          np.mean([
              len(recommend_cf(user, n=10))
              for user in test_user_items
          ]), 2
      ))

Users with relevant test recipes: 996
Average relevant recipes per user: 12.52
Average recommendations per user: 9.94


In [16]:
# Keep only relevant test recipes that exist in the training catalogue
test_relevant_eval = test_relevant[
    test_relevant["recipe_id"].isin(recipe_map)
].copy()

print("Original relevant test interactions:", len(test_relevant))
print("Relevant interactions available for evaluation:", len(test_relevant_eval))
print("Users with evaluable relevant recipes:",
      test_relevant_eval["user_id"].nunique())

Original relevant test interactions: 12472
Relevant interactions available for evaluation: 5382
Users with evaluable relevant recipes: 898


In [17]:
def calculate_metrics(recommendation_function, k=10):
    precisions = []
    recalls = []

    relevant_by_user = (
        test_relevant_eval.groupby("user_id")["recipe_id"]
        .apply(set).to_dict()
    )

    for user_id, relevant_set in relevant_by_user.items():

        recommendations = recommendation_function(user_id, n=k)

        recommended_set = set(recommendations)

        hits = len(recommended_set & relevant_set)

        precision = hits / k
        recall = hits / len(relevant_set)

        precisions.append(precision)
        recalls.append(recall)

    return {
        "Precision@10": round(float(np.mean(precisions)), 4),
        "Recall@10": round(float(np.mean(recalls)), 4)
    }

In [18]:
cf_results = calculate_metrics(recommend_cf)

print("Updated Collaborative Filtering Results:")
print(cf_results)

Updated Collaborative Filtering Results:
{'Precision@10': 0.0003, 'Recall@10': 0.0004}


In [19]:
# Relevant test recipes that also exist in the training catalogue
relevant_in_catalog = test_relevant[
    test_relevant["recipe_id"].isin(recipe_map)
]

print("Total relevant test interactions:", len(test_relevant))

print("Relevant test interactions in training catalogue:",
      len(relevant_in_catalog))

print("Percentage covered:",
      round(len(relevant_in_catalog) / len(test_relevant) * 100, 2), "%")

# Check how many relevant recipes each user has available
print("\nAverage relevant recipes per user in catalogue:",
      round(relevant_in_catalog.groupby("user_id").size().mean(), 2))

Total relevant test interactions: 12472
Relevant test interactions in training catalogue: 5382
Percentage covered: 43.15 %

Average relevant recipes per user in catalogue: 5.99


In [20]:
total_hits = 0
total_recommendations = 0

for user_id in test_user_items:
    recommendations = set(recommend_cf(user_id, n=10))

    actual = set(
        relevant_in_catalog[
            relevant_in_catalog["user_id"] == user_id
        ]["recipe_id"]
    )

    total_hits += len(recommendations & actual)
    total_recommendations += len(recommendations)

print("Total hits:", total_hits)
print("Total recommendations:", total_recommendations)

Total hits: 3
Total recommendations: 9937


In [21]:

def calculate_metrics(recommendation_function, k=10):
    precisions = []
    recalls = []

    for user_id in test_user_items:

        actual_relevant = relevant_in_catalog[
            relevant_in_catalog["user_id"] == user_id
        ]["recipe_id"].tolist()

        if len(actual_relevant) == 0:
            continue

        recommendations = recommendation_function(user_id, n=k)

        recommended_set = set(recommendations)
        relevant_set = set(actual_relevant)

        hits = len(recommended_set & relevant_set)

        precision = hits / k
        recall = hits / len(relevant_set)

        precisions.append(precision)
        recalls.append(recall)

    return {
        "Precision@10": round(float(np.mean(precisions)), 4),
        "Recall@10": round(float(np.mean(recalls)), 4)
    }


In [22]:
cf_results = calculate_metrics(recommend_cf)

print("Corrected Collaborative Filtering Results:")
print(cf_results)

Corrected Collaborative Filtering Results:
{'Precision@10': 0.0003, 'Recall@10': 0.0004}


In [23]:
print("Preparing Case-Based Filtering...")

# Convert recipe ingredients and tags into lists
def parse_list(value):
    try:
        return ast.literal_eval(value) if isinstance(value, str) else []
    except (ValueError, SyntaxError):
        return []

recipes_clean["ingredients_list"] = recipes_clean["ingredients"].apply(parse_list)
recipes_clean["tags_list"] = recipes_clean["tags"].apply(parse_list)

# Create combined recipe descriptions
recipes_clean["case_text"] = (
    recipes_clean["name"].fillna("") + " " +
    recipes_clean["description"].fillna("") + " " +
    recipes_clean["ingredients_list"].apply(lambda x: " ".join(x)) + " " +
    recipes_clean["tags_list"].apply(lambda x: " ".join(x))
)

print("Recipe case descriptions created!")
print("Total recipe cases:", len(recipes_clean))

Preparing Case-Based Filtering...
Recipe case descriptions created!
Total recipe cases: 231637


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

print("Converting recipe cases into numerical vectors...")

case_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    min_df=2,
    max_df=0.90,
    dtype=np.float32
)

case_vectors = case_vectorizer.fit_transform(
    recipes_clean["case_text"]
)

print("Case vectors created:", case_vectors.shape)

print("\nTraining Case-Based model...")

case_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)

case_model.fit(case_vectors)

print("Case-Based model trained successfully!")

Converting recipe cases into numerical vectors...
Case vectors created: (231637, 37417)

Training Case-Based model...
Case-Based model trained successfully!


In [25]:
recipe_id_to_row = {
    recipe_id: index
    for index, recipe_id in enumerate(recipes_clean["id"])
}

def recommend_case_for_user(user_id, n=10):

    user_history = train_ratings[
        train_ratings["user_id"] == user_id
    ]

    liked_recipes = user_history[
        user_history["rating"] >= 4
    ]["recipe_id"].tolist()

    if len(liked_recipes) == 0:
        liked_recipes = user_history["recipe_id"].tolist()

    liked_rows = [
        recipe_id_to_row[rid]
        for rid in liked_recipes
        if rid in recipe_id_to_row
    ]

    if len(liked_rows) == 0:
        return []

    user_profile = np.asarray(
        case_vectors[liked_rows].mean(axis=0)
    )

    similarities = cosine_similarity(
        user_profile, case_vectors
    ).flatten()

    # Exclude recipes already rated by this user
    rated_recipes = set(user_history["recipe_id"])

    candidate_indices = np.argsort(similarities)[::-1]

    recommendations = []

    for index in candidate_indices:
        recipe_id = recipes_clean.iloc[index]["id"]

        if recipe_id not in rated_recipes:
            recommendations.append(recipe_id)

        if len(recommendations) == n:
            break

    return recommendations

print("Case-Based recommendation function created!")

Case-Based recommendation function created!


In [26]:
case_results = calculate_metrics(
    recommend_case_for_user,
    k=10
)

print("Case-Based Filtering Results:")
print(case_results)

Case-Based Filtering Results:
{'Precision@10': 0.0012, 'Recall@10': 0.0029}


In [27]:
def normalize_scores(scores):
    scores = np.array(scores, dtype=float)

    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros_like(scores)

    return (scores - min_score) / (max_score - min_score)


def recommend_hybrid(user_id, n=10):

    if user_id not in user_map:
        return []

    # Collaborative Filtering scores
    user_index = user_map[user_id]
    user_ratings = sparse_matrix[user_index].toarray().flatten()

    rated_indices = np.where(user_ratings > 0)[0]

    cf_scores = np.zeros(len(recipe_ids))

    for recipe_index in rated_indices:
        rating = user_ratings[recipe_index]

        similar_indices = neighbor_indices[recipe_index][1:]
        similar_distances = distances[recipe_index][1:]

        similarities = 1 - similar_distances

        cf_scores[similar_indices] += rating * similarities

    # Case-Based scores
    user_history = train_ratings[
        train_ratings["user_id"] == user_id
    ]

    liked_recipes = user_history[
        user_history["rating"] >= 4
    ]["recipe_id"].tolist()

    if len(liked_recipes) == 0:
        liked_recipes = user_history["recipe_id"].tolist()

    liked_rows = [
        recipe_id_to_row[rid]
        for rid in liked_recipes
        if rid in recipe_id_to_row
    ]

    if len(liked_rows) == 0:
        return []

    user_profile = np.asarray(
        case_vectors[liked_rows].mean(axis=0)
    )

    case_scores = cosine_similarity(
        user_profile, case_vectors
    ).flatten()

    # Combine scores for recipes in the training catalogue
    candidate_indices = np.where(user_ratings == 0)[0]

    candidate_ids = recipe_ids[candidate_indices]

    candidate_cf = normalize_scores(cf_scores[candidate_indices])

    candidate_case = normalize_scores([
        case_scores[recipe_id_to_row[rid]]
        for rid in candidate_ids
    ])

    hybrid_scores = (
        0.5 * candidate_cf +
        0.5 * candidate_case
    )

    top_indices = np.argsort(hybrid_scores)[::-1][:n]

    return candidate_ids[top_indices].tolist()


print("Hybrid recommendation function created!")

Hybrid recommendation function created!


In [28]:
hybrid_results = calculate_metrics(
    recommend_hybrid,
    k=10
)

print("Hybrid Recommendation Results:")
print(hybrid_results)

Hybrid Recommendation Results:
{'Precision@10': 0.0012, 'Recall@10': 0.0035}


In [29]:
comparison = pd.DataFrame({
    "Model": [
        "Collaborative Filtering",
        "Case-Based Filtering",
        "Hybrid Recommendation"
    ],
    "Precision@10": [
        cf_results["Precision@10"],
        case_results["Precision@10"],
        hybrid_results["Precision@10"]
    ],
    "Recall@10": [
        cf_results["Recall@10"],
        case_results["Recall@10"],
        hybrid_results["Recall@10"]
    ]
})

comparison = comparison.sort_values(
    by="Recall@10",
    ascending=False
).reset_index(drop=True)

print("FINAL MODEL COMPARISON")
display(comparison)

FINAL MODEL COMPARISON


,Model,Precision@10,Recall@10
0,Hybrid Recommendation,0.0012,0.0035
1,Case-Based Filtering,0.0012,0.0029
2,Collaborative Filtering,0.0003,0.0004


In [30]:
comparison.to_csv(
    "../results/model_comparison.csv",
    index=False
)

print("Model comparison saved successfully!")

Model comparison saved successfully!
